In [ ]:
import sys
import math

# Please do not remove package declarations because these are used by the autograder.

import sys
from collections import defaultdict

def profile_hmm(theta: float,
                alphabet: list[str],
                alignment: list[str]) -> tuple[list[str], list[str], list[list[float]], list[list[float]]]:

    n_seq = len(alignment)
    n_col = len(alignment[0])

    
    # A column is a MATCH column if fraction of gaps < theta.
    # Otherwise it is an INSERT column.
    is_match = []
    for c in range(n_col):
        gaps = sum(1 for seq in alignment if seq[c] == '-')
        # print(gaps)
        # print(gaps / n_seq)
        is_match.append(gaps / n_seq < theta)

    n_match = sum(is_match)   # number of match columns

    
    match_num = []  
    m = 0
    for c in range(n_col):
        if is_match[c]:
            m += 1
            match_num.append(m)
        else:
            match_num.append(None)

    
    # S, I0, M1,D1,I1, M2,D2,I2, ..., ML,DL,IL, E
    states = ['S', 'I0']
    for i in range(1, n_match + 1):
        states += [f'M{i}', f'D{i}', f'I{i}']
    states.append('E')

    state_idx = {s: i for i, s in enumerate(states)} #{'S': 0, 'I0': 1, 'M1': 2, 'D1': 3, 'I1': 4, 'M2': 5, 'D2': 6, 'I2': 7, 'E': 8}

    # print(state_idx)
    n_states = len(states)
    alph_idx = {a: i for i, a in enumerate(alphabet)} # {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    # print(alph_idx)
    n_alpha  = len(alphabet)


    trans_counts = defaultdict(lambda: defaultdict(float))
    emit_counts  = defaultdict(lambda: defaultdict(float))
    # print(trans_counts,emit_counts)
    #print(is_match)
    for seq in alignment:
        # Build the ordered list of (state, emitted_char_or_None) for this sequence
        path = ['S']
        cur_match = 0  # last match column number we've "consumed"

        for c in range(n_col):
            ch = seq[c]
            if is_match[c]:
                # This is a match column
                cur_match += 1
                if ch == '-':
                    path.append(f'D{cur_match}')
                else:
                    path.append(f'M{cur_match}')
            else:
                # Insert column — only emit if not a gap
                if ch != '-':
                    # The insert state belongs to the current match number
                    # (i.e., I{cur_match}, which is I0 before any match col)
                    path.append(f'I{cur_match}')

        path.append('E')
        # print(path)list of lists (with all emissions and their states in order)

        # Count transitions probably the most difficult part
        for i in range(len(path) - 1):
            trans_counts[path[i]][path[i+1]] += 1
        #print(trans_counts)
        # Count emissions: walk path and alignment columns together
        match_ptr = 0   # which match column we're on
        insert_ptr = 0  # which insert column we're on (within current insert block)

        # Re-trace to pair states with characters
        state_pos = 0  # index into path (skip S at 0)
        col = 0
        for state in path[1:-1]:  # skip S and E
            if state.startswith('M'):
                # find next match column
                while not is_match[col]:
                    col += 1
                ch = seq[col]
                emit_counts[state][ch] += 1
                col += 1
            elif state.startswith('D'):
                # find next match column (gap there)
                while not is_match[col]:
                    col += 1
                col += 1   # skip it (gap, no emission)
            elif state.startswith('I'):
                # find next non-gap insert column
                while col < n_col and is_match[col]:
                    col += 1
                # now col is an insert column
                ch = seq[col]
                emit_counts[state][ch] += 1
                col += 1

   
    transition = [[0.0] * n_states for _ in range(n_states)]
    emission   = [[0.0] * n_alpha  for _ in range(n_states)]

    for s, nbrs in trans_counts.items():
        total = sum(nbrs.values())
        si = state_idx[s]
        for t, cnt in nbrs.items():
            transition[si][state_idx[t]] = round(cnt / total, 10)

    for s, chars in emit_counts.items():
        total = sum(chars.values())
        si = state_idx[s]
        for ch, cnt in chars.items():
            emission[si][alph_idx[ch]] = round(cnt / total, 10)

    return states, alphabet, transition, emission




In [ ]:
# Profile HMM Problem: Construct a profile HMM from a multiple alignment.

# Input: A multiple alignment Alignment and a threshold θ.
# Output: HMM(Alignment, θ), in the form of transition and emission matrices.
# Code Challenge: Solve the Profile HMM Problem.

# Input: A threshold θ, followed by an alphabet Σ, followed by a multiple alignment Alignment whose strings are formed from Σ.
# Output: The profile HMM HMM(Alignment, θ) represented by its states, alphabet, transition matrix, and emission matrix.
# Note: The rows and columns of the transition matrix and the emission matrix must correspond exactly to the order in which you output your states and alphabet.

# Debug Datasets

# Sample Input 1:

# 0.289
# --------
# A B C D E
# --------
# EBA
# E-D
# EB-
# EED
# EBD
# EBE
# E-D
# E-D
# Sample Output 1:

# S I0 M1 D1 I1 M2 D2 I2 E
# --------
# A B C D E
# --------
# 	S	I0	M1	D1	I1	M2	D2	I2	E
# S	0	0	1.0	0	0	0	0	0	0
# I0	0	0	0	0	0	0	0	0	0
# M1	0	0	0	0	0.625	0.375	0	0	0
# D1	0	0	0	0	0	0	0	0	0
# I1	0	0	0	0	0	0.8	0.2	0	0
# M2	0	0	0	0	0	0	0	0	1.0
# D2	0	0	0	0	0	0	0	0	1.0
# I2	0	0	0	0	0	0	0	0	0
# E	0	0	0	0	0	0	0	0	0
# --------
# 	A	B	C	D	E
# S	0	0	0	0	0
# I0	0	0	0	0	0
# M1	0	0	0	0	1.0
# D1	0	0	0	0	0
# I1	0	0.8	0	0	0.2
# M2	0.143	0	0	0.714	0.143
# D2	0	0	0	0	0
# I2	0	0	0	0	0
# E	0	0	0	0	0
# Sample Input 2:

# 0.1
# --------
# A B
# --------
# AA
# AA
# Sample Output 2:

# S I0 M1 D1 I1 M2 D2 I2 E
# --------
# A B
# --------
# 	S	I0	M1	D1	I1	M2	D2	I2	E
# S	0	0	1.0	0	0	0	0	0	0
# I0	0	0	0	0	0	0	0	0	0
# M1	0	0	0	0	0	1.0	0	0	0
# D1	0	0	0	0	0	0	0	0	0
# I1	0	0	0	0	0	0	0	0	0
# M2	0	0	0	0	0	0	0	0	1.0
# D2	0	0	0	0	0	0	0	0	0
# I2	0	0	0	0	0	0	0	0	0
# E	0	0	0	0	0	0	0	0	0
# --------
# 	A	B
# S	0	0
# I0	0	0
# M1	1.0	0
# D1	0	0
# I1	0	0
# M2	1.0	0
# D2	0	0
# I2	0	0
# E	0	0
# Sample Input 3:

# 0.4
# --------
# A B
# --------
# AB
# A-
# Sample Output 3:

# S I0 M1 D1 I1 E
# --------
# A B
# --------
# 	S	I0	M1	D1	I1	E
# S	0	0	1.0	0	0	0
# I0	0	0	0	0	0	0
# M1	0	0	0	0	0.5	0.5
# D1	0	0	0	0	0	0
# I1	0	0	0	0	0	1.0
# E	0	0	0	0	0	0
# --------
# 	A	B
# S	0	0
# I0	0	0
# M1	1.0	0
# D1	0	0
# I1	0	1.0
# E	0	0
